In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1G7JrHxlCObjSljvWvaA23GeFUo-gXUt6", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/01_00_intro.mp3"))


# Notebook 1: Mixture of Experts from Scratch -- Vizuara

In this notebook, we will build a **Mixture of Experts (MoE)** layer entirely from scratch using PyTorch. By the end, you will understand exactly how tokens are routed to a subset of expert MLPs, how the outputs are combined, and why MoE achieves massive model capacity with sparse computation.

**What you will build:**
- A router (gating network) that assigns tokens to experts
- Multiple independent expert MLP sub-networks
- A complete MoE layer with top-k routing and weighted combination
- A mini transformer block that uses MoE instead of a standard MLP

**Estimated time:** 40--60 minutes

**Prerequisites:** Basic PyTorch, understanding of transformer MLPs

In [ ]:
#@title 🎧 Code Walkthrough: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_01_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Setup: Run this cell first!
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("No GPU detected. This notebook runs fine on CPU.")
    print("Go to Runtime -> Change runtime type -> GPU if you want GPU.")

print(f"Python {sys.version.split()[0]}")
print(f"PyTorch {torch.__version__}")

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"Random seed: {SEED}")

%matplotlib inline

In [ ]:
#@title 🎧 Listen: Standard Mlp Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_02_standard_mlp_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 1. The Standard Transformer MLP

Before building MoE, let us be clear about what it replaces. In a standard transformer block, the MLP consists of two linear projections with an activation in between:

$$\text{MLP}(\mathbf{x}) = \mathbf{W}_2 \, \sigma(\mathbf{W}_1 \mathbf{x})$$

where $\mathbf{W}_1 \in \mathbb{R}^{4d \times d}$ projects up and $\mathbf{W}_2 \in \mathbb{R}^{d \times 4d}$ projects back down.

This is a **dense** model -- every token activates every parameter. MoE replaces this single MLP with multiple expert MLPs plus a router.

In [ ]:
#@title 🎧 Code Walkthrough: Standard Mlp Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_03_standard_mlp_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class StandardMLP(nn.Module):
    """The standard transformer MLP that MoE replaces."""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.act = nn.GELU()

    def forward(self, x):
        return self.w2(self.act(self.w1(x)))

# Example: d_model=512, d_ff=2048
d_model = 512
d_ff = 2048
mlp = StandardMLP(d_model, d_ff)

total_params = sum(p.numel() for p in mlp.parameters())
print(f"Standard MLP parameters: {total_params:,}")
print(f"  W1: {d_model} x {d_ff} = {d_model * d_ff:,}")
print(f"  W2: {d_ff} x {d_model} = {d_ff * d_model:,}")
print(f"  + biases: {d_ff + d_model:,}")
print(f"\nEvery token activates ALL {total_params:,} parameters.")

In [ ]:
#@title 🎧 Listen: Router Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_04_router_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. Building the Router

The **router** (also called the gating network) is the decision-maker. For each token, it computes a probability distribution over $N$ experts and selects the top-$k$.

The router is simply a linear layer followed by softmax:

$$\mathbf{h} = \mathbf{W}_r \mathbf{x}, \quad p_i = \frac{e^{h_i}}{\sum_{j=1}^{N} e^{h_j}}$$

Then we select the top-$k$ experts and renormalize their weights.

In [ ]:
#@title 🎧 Code Walkthrough: Router Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_05_router_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class Router(nn.Module):
    """Token router: assigns each token to its top-k experts."""
    def __init__(self, d_model, num_experts, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        # Single linear layer: maps d_model -> num_experts
        self.gate = nn.Linear(d_model, num_experts, bias=False)

    def forward(self, x):
        """
        Args:
            x: (batch, seq_len, d_model)
        Returns:
            top_k_weights: (batch, seq_len, top_k) -- renormalized weights
            top_k_indices: (batch, seq_len, top_k) -- expert indices
            router_probs: (batch, seq_len, num_experts) -- full probabilities
        """
        # Compute logits and probabilities
        logits = self.gate(x)                           # (B, S, N)
        router_probs = F.softmax(logits, dim=-1)        # (B, S, N)

        # Select top-k experts
        top_k_probs, top_k_indices = torch.topk(
            router_probs, self.top_k, dim=-1
        )  # both: (B, S, top_k)

        # Renormalize so the top-k weights sum to 1
        top_k_weights = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)

        return top_k_weights, top_k_indices, router_probs


# Test the router
num_experts = 8
top_k = 2
router = Router(d_model, num_experts, top_k)

# Create a batch of tokens
batch_size, seq_len = 2, 4
x = torch.randn(batch_size, seq_len, d_model)

weights, indices, probs = router(x)
print(f"Input shape: {x.shape}")
print(f"Router probs shape: {probs.shape}")
print(f"Top-k weights shape: {weights.shape}")
print(f"Top-k indices shape: {indices.shape}")

# Show routing for first token
print(f"\nFirst token routing:")
print(f"  Full probabilities: {probs[0, 0].detach().numpy().round(3)}")
print(f"  Selected experts: {indices[0, 0].tolist()}")
print(f"  Renormalized weights: {weights[0, 0].detach().numpy().round(3)}")
print(f"  Weights sum to: {weights[0, 0].sum().item():.4f}")

In [ ]:
#@title 🎧 Listen: Router Viz Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_06_router_viz_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Visualizing Router Decisions

Let us see how the router distributes tokens across experts.

In [ ]:
#@title 🎧 What to Look For: Router Viz Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_07_router_viz_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Generate many tokens and see the routing distribution
torch.manual_seed(42)
many_tokens = torch.randn(1, 1000, d_model)
_, many_indices, many_probs = router(many_tokens)

# Count how many tokens are routed to each expert
expert_counts = torch.zeros(num_experts)
for k in range(top_k):
    for e in range(num_experts):
        expert_counts[e] += (many_indices[0, :, k] == e).sum().item()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of token counts per expert
colors = ['#2196F3' if c < expert_counts.mean() * 1.5 else '#f44336'
          for c in expert_counts]
axes[0].bar(range(num_experts), expert_counts.numpy(), color=colors, edgecolor='white')
axes[0].axhline(y=1000 * top_k / num_experts, color='gray', linestyle='--',
                label=f'Ideal: {1000 * top_k / num_experts:.0f} tokens')
axes[0].set_xlabel('Expert Index', fontsize=12)
axes[0].set_ylabel('Number of Tokens Routed', fontsize=12)
axes[0].set_title('Token Distribution Across Experts (Untrained Router)', fontsize=13)
axes[0].legend(fontsize=11)

# Heatmap of router probabilities for first 20 tokens
im = axes[1].imshow(many_probs[0, :20].detach().numpy(), aspect='auto',
                     cmap='Blues')
axes[1].set_xlabel('Expert Index', fontsize=12)
axes[1].set_ylabel('Token Index', fontsize=12)
axes[1].set_title('Router Probabilities (first 20 tokens)', fontsize=13)
plt.colorbar(im, ax=axes[1], label='Probability')

plt.tight_layout()
plt.show()

print(f"\nToken count per expert: {expert_counts.numpy().astype(int)}")
print(f"Ideal (balanced): {1000 * top_k / num_experts:.0f} tokens each")
print(f"Std deviation: {expert_counts.std().item():.1f}")

In [ ]:
#@title 🎧 Listen: Expert Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_08_expert_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. Building an Expert MLP

Each expert is an independent MLP -- structurally identical to the standard transformer MLP, but each one has its own learned weights. With $N$ experts, we have $N$ separate copies of the MLP.

In [ ]:
#@title 🎧 Code Walkthrough: Expert Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_09_expert_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class Expert(nn.Module):
    """A single expert MLP (same structure as standard transformer MLP)."""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.act = nn.GELU()

    def forward(self, x):
        # x: (num_tokens, d_model)
        return self.w2(self.act(self.w1(x)))

# Create 8 experts
experts = nn.ModuleList([Expert(d_model, d_ff) for _ in range(num_experts)])

single_expert_params = sum(p.numel() for p in experts[0].parameters())
total_expert_params = sum(p.numel() for p in experts.parameters())
print(f"Parameters per expert: {single_expert_params:,}")
print(f"Total expert parameters ({num_experts} experts): {total_expert_params:,}")
print(f"\nWith top-k={top_k}, active params per token: ~{single_expert_params * top_k:,}")
print(f"That is {top_k}/{num_experts} = {top_k/num_experts*100:.0f}% of total expert params")

In [ ]:
#@title 🎧 Listen: Moe Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_10_moe_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. The Complete MoE Layer

Now we combine the router and experts into a complete **MoE layer**. The forward pass:

1. Router computes probabilities and selects top-k experts per token
2. Each selected expert processes only the tokens routed to it
3. Outputs are combined using the renormalized router weights

$$\mathbf{y} = \sum_{i \in \text{TopK}} \tilde{p}_i \cdot \text{Expert}_i(\mathbf{x})$$

In [ ]:
#@title 🎧 Code Walkthrough: Moe Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_11_moe_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class MoELayer(nn.Module):
    """Complete Mixture of Experts layer."""
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.router = Router(d_model, num_experts, top_k)
        self.experts = nn.ModuleList([
            Expert(d_model, d_ff) for _ in range(num_experts)
        ])

    def forward(self, x):
        """
        Args:
            x: (batch, seq_len, d_model)
        Returns:
            output: (batch, seq_len, d_model)
            router_probs: (batch, seq_len, num_experts) -- for load balancing
        """
        batch, seq_len, d = x.shape

        # Step 1: Get routing decisions
        weights, indices, router_probs = self.router(x)
        # weights: (B, S, top_k), indices: (B, S, top_k)

        # Step 2: Dispatch tokens to experts and combine outputs
        output = torch.zeros_like(x)

        for k_idx in range(self.top_k):
            expert_idx = indices[:, :, k_idx]              # (B, S)
            weight = weights[:, :, k_idx].unsqueeze(-1)    # (B, S, 1)

            for e in range(self.num_experts):
                mask = (expert_idx == e)                    # (B, S)
                if mask.any():
                    # Gather tokens for this expert
                    tokens = x[mask]                        # (num_selected, d)
                    # Run expert
                    expert_out = self.experts[e](tokens)    # (num_selected, d)
                    # Scatter weighted output back
                    output[mask] += weight[mask] * expert_out

        return output, router_probs


# Test the MoE layer
moe = MoELayer(d_model, d_ff, num_experts=8, top_k=2)

x = torch.randn(2, 4, d_model)
out, probs = moe(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Probs shape:  {probs.shape}")

total_moe_params = sum(p.numel() for p in moe.parameters())
print(f"\nTotal MoE layer params: {total_moe_params:,}")
print(f"Standard MLP params:    {total_params:,}")
print(f"Ratio: {total_moe_params / total_params:.1f}x more params in MoE")
print(f"But only {top_k}/{num_experts} = {top_k/num_experts*100:.0f}% active per token")

In [ ]:
#@title 🎧 Listen: Numerical Walkthrough Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_12_numerical_walkthrough_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. Numerical Walkthrough

Let us trace through the routing computation step by step, matching the article's example.

In [ ]:
#@title 🎧 Code Walkthrough: Numerical Walkthrough Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_13_numerical_walkthrough_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Reproduce the article's numerical example
# Router scores: h = [1.2, 0.3, 2.5, 0.1, 0.8, 3.1, 0.2, 0.5]
h = torch.tensor([1.2, 0.3, 2.5, 0.1, 0.8, 3.1, 0.2, 0.5])

# Step 1: Softmax
p = F.softmax(h, dim=0)
print("Step 1: Softmax probabilities")
for i, pi in enumerate(p):
    print(f"  Expert {i}: h={h[i]:.1f} -> p={pi:.4f}")

# Step 2: Top-k selection (k=2)
k = 2
top_vals, top_idx = torch.topk(p, k)
print(f"\nStep 2: Top-{k} selection")
print(f"  Selected experts: {top_idx.tolist()}")
print(f"  Their probabilities: {top_vals.tolist()}")

# Step 3: Renormalize
renorm = top_vals / top_vals.sum()
print(f"\nStep 3: Renormalized weights")
for idx, w in zip(top_idx, renorm):
    print(f"  Expert {idx.item()}: {w:.4f}")
print(f"  Sum: {renorm.sum():.4f}")

print(f"\nFinal output: y = {renorm[0]:.3f} * Expert_{top_idx[0]}(x) + {renorm[1]:.3f} * Expert_{top_idx[1]}(x)")
print(f"The other {num_experts - k} experts are NEVER computed for this token.")

In [ ]:
#@title 🎧 Listen: Tradeoff Concept
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_14_tradeoff_concept.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. The Parameter vs. Compute Trade-Off

The magic of MoE: huge total capacity but sparse compute per token.

$$\text{Active FLOPs per token} \approx \frac{k}{N} \times \text{Total expert FLOPs} + \text{Shared FLOPs}$$

In [ ]:
#@title 🎧 Code Walkthrough: Tradeoff Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_15_tradeoff_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Compare MoE vs dense models at different scales
configs = [
    {"name": "Dense 7B (baseline)", "experts": 1, "expert_size": "7B",
     "total_params": 7, "top_k": 1, "active_frac": 1.0},
    {"name": "Mixtral 8x7B", "experts": 8, "expert_size": "7B",
     "total_params": 47, "top_k": 2, "active_frac": 2/8},
    {"name": "Switch-C (128 experts)", "experts": 128, "expert_size": "~0.2B",
     "total_params": 1571, "top_k": 1, "active_frac": 1/128},
    {"name": "DeepSeek-V3", "experts": 256, "expert_size": "~0.4B",
     "total_params": 671, "top_k": 8, "active_frac": 8/256},
]

print(f"{'Model':<25} {'Experts':>8} {'Top-k':>6} {'Total (B)':>10} {'Active %':>9} {'Equiv Dense (B)':>15}")
print("-" * 80)
for c in configs:
    equiv = c['total_params'] * c['active_frac']
    print(f"{c['name']:<25} {c['experts']:>8} {c['top_k']:>6} {c['total_params']:>10.0f} "
          f"{c['active_frac']*100:>8.1f}% {equiv:>15.1f}")

print("\nKey insight: MoE models have the CAPACITY of their total params")
print("but the COMPUTE COST of a much smaller dense model.")

## 7. Your Turn -- TODO Exercises


In [ ]:
#@title 🎧 Before You Start: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_16_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO 1: Implement a MoE Transformer Block

Build a transformer block that uses MoE instead of a standard MLP. The block should have:
1. Self-attention (provided)
2. LayerNorm (provided)
3. MoE layer (your task)
4. Residual connections

In [ ]:
#@title 🎧 Before You Start: Todo1 Post
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_17_todo1_post.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class MoETransformerBlock(nn.Module):
    """
    Transformer block with MoE replacing the standard MLP.

    Structure:
      x -> LayerNorm -> Self-Attention -> + (residual)
        -> LayerNorm -> MoE Layer      -> + (residual)
    """
    def __init__(self, d_model, n_heads, d_ff, num_experts, top_k):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)

        # ============ TODO ============
        # Create the MoE layer using the MoELayer class defined above
        self.moe = ???  # YOUR CODE HERE
        # ==============================

    def forward(self, x):
        # Self-attention with residual
        normed = self.ln1(x)
        attn_out, _ = self.attn(normed, normed, normed)
        x = x + attn_out

        # ============ TODO ============
        # MoE with residual connection
        # 1. Apply LayerNorm (self.ln2)
        # 2. Pass through MoE layer (returns output and router_probs)
        # 3. Add residual connection
        normed = ???       # YOUR CODE HERE
        moe_out, probs = ???  # YOUR CODE HERE
        x = ???            # YOUR CODE HERE
        # ==============================

        return x, probs


# Test your implementation
block = MoETransformerBlock(
    d_model=512, n_heads=8, d_ff=2048, num_experts=8, top_k=2
)
x = torch.randn(2, 16, 512)
out, probs = block(x)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")
print(f"Probs:  {probs.shape}")
assert out.shape == x.shape, "Output shape should match input shape!"
print("\nTODO 1 passed!")

In [ ]:
#@title 🎧 Before You Start: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_18_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO 2: Compute Expert Utilization Statistics

Given a batch of tokens and their routing decisions, compute:
- $f_i$: fraction of tokens routed to each expert
- $P_i$: average router probability for each expert
- The coefficient of variation (std/mean) as a measure of imbalance

In [ ]:
#@title 🎧 Before You Start: Todo2 Post
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_19_todo2_post.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def compute_expert_utilization(router_probs, top_k_indices, num_experts):
    """
    Compute expert utilization statistics.

    Args:
        router_probs: (batch, seq_len, num_experts) -- full softmax probs
        top_k_indices: (batch, seq_len, top_k) -- selected expert indices
        num_experts: int

    Returns:
        f_i: (num_experts,) -- fraction of tokens routed to each expert
        P_i: (num_experts,) -- average router probability per expert
        cv: float -- coefficient of variation of f_i
    """
    B, S, K = top_k_indices.shape
    total_assignments = B * S * K  # Each token makes K assignments

    # ============ TODO ============
    # Compute f_i: for each expert, count how many times it appears
    # in top_k_indices and divide by total_assignments
    f_i = ???  # YOUR CODE HERE -- shape: (num_experts,)

    # Compute P_i: average of router_probs[:, :, i] across all tokens
    P_i = ???  # YOUR CODE HERE -- shape: (num_experts,)

    # Compute coefficient of variation of f_i
    cv = ???  # YOUR CODE HERE -- scalar
    # ==============================

    return f_i, P_i, cv


# Test with our MoE layer
torch.manual_seed(42)
test_x = torch.randn(4, 128, d_model)  # 4 batches, 128 tokens each
_, test_probs = moe(test_x)
_, test_indices, _ = moe.router(test_x)

f_i, P_i, cv = compute_expert_utilization(test_probs, test_indices, num_experts)

print("Expert utilization (untrained router):")
print(f"{'Expert':>8} {'f_i':>10} {'P_i':>10}")
print("-" * 30)
for i in range(num_experts):
    print(f"{i:>8} {f_i[i].item():>10.4f} {P_i[i].item():>10.4f}")
print(f"\nCoefficient of variation: {cv:.4f}")
print(f"(0 = perfectly balanced, higher = more imbalanced)")

In [ ]:
#@title 🎧 Before You Start: Todo3 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_20_todo3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO 3: Implement Expert Choice Routing

Instead of tokens choosing experts, let **experts choose tokens**. Each expert selects its top-$k_e$ tokens, guaranteeing perfect load balance.

In [ ]:
#@title 🎧 Before You Start: Todo3 Post
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_21_todo3_post.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class ExpertChoiceRouter(nn.Module):
    """Expert Choice routing: experts choose their tokens."""
    def __init__(self, d_model, num_experts, capacity_factor=1.0):
        super().__init__()
        self.num_experts = num_experts
        self.capacity_factor = capacity_factor
        self.gate = nn.Linear(d_model, num_experts, bias=False)

    def forward(self, x):
        """
        Args:
            x: (batch, seq_len, d_model)
        Returns:
            dispatch_mask: (num_experts, tokens_per_expert) -- indices of tokens
            expert_weights: (num_experts, tokens_per_expert) -- weights for each token
        """
        B, S, d = x.shape
        T = B * S  # Total tokens
        tokens_per_expert = int(self.capacity_factor * T / self.num_experts)

        # Reshape to (T, d)
        x_flat = x.reshape(T, d)

        # ============ TODO ============
        # 1. Compute router logits: (T, num_experts)
        logits = ???  # YOUR CODE HERE

        # 2. Apply softmax across tokens (dim=0), so each expert
        #    gets a probability distribution over all tokens
        probs = ???  # YOUR CODE HERE -- softmax over dim=0

        # 3. Transpose to (num_experts, T) so each row is one expert's
        #    probability distribution over tokens
        expert_probs = ???  # YOUR CODE HERE

        # 4. Each expert selects its top-tokens_per_expert tokens
        expert_weights, dispatch_mask = torch.topk(
            expert_probs, tokens_per_expert, dim=-1
        )  # both: (num_experts, tokens_per_expert)
        # ==============================

        return dispatch_mask, expert_weights, tokens_per_expert


# Test Expert Choice routing
ec_router = ExpertChoiceRouter(d_model, num_experts=8, capacity_factor=1.0)
x_test = torch.randn(2, 32, d_model)  # 64 total tokens

dispatch, weights_ec, tpe = ec_router(x_test)
print(f"Total tokens: {2 * 32}")
print(f"Tokens per expert: {tpe}")
print(f"Dispatch mask shape: {dispatch.shape}")
print(f"Expert weights shape: {weights_ec.shape}")
print(f"\nEvery expert gets EXACTLY {tpe} tokens -- perfect balance!")

# Check: how many experts process each token?
all_token_ids = dispatch.reshape(-1)
unique, counts = torch.unique(all_token_ids, return_counts=True)
print(f"\nTokens processed by 0 experts: {64 - len(unique)}")
print(f"Tokens processed by 1 expert: {(counts == 1).sum().item()}")
print(f"Tokens processed by 2+ experts: {(counts >= 2).sum().item()}")

In [ ]:
#@title 🎧 Listen: Training Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_22_training_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 8. Comparing MoE vs Dense: A Training Experiment

Let us train a small MoE model vs a dense model of equivalent active parameters on a simple task to see the capacity advantage.

In [ ]:
#@title 🎧 What to Look For: Training Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_23_training_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Simple task: predict the XOR pattern in sequences
def generate_xor_data(n_samples, seq_len, d_model):
    """Generate data where the target depends on XOR of input features."""
    x = torch.randn(n_samples, seq_len, d_model)
    # Target: sign of product of first two features
    target = (x[:, :, 0] * x[:, :, 1] > 0).float().unsqueeze(-1)
    target = target.expand(-1, -1, d_model)  # Expand to d_model
    return x, target

# Train both models
d = 64  # Small dimension for speed
d_ff_small = 256

dense_model = StandardMLP(d, d_ff_small)
moe_model = MoELayer(d, d_ff_small, num_experts=4, top_k=1)

dense_params = sum(p.numel() for p in dense_model.parameters())
moe_params = sum(p.numel() for p in moe_model.parameters())
print(f"Dense model params: {dense_params:,}")
print(f"MoE model params:   {moe_params:,} (but only 1/4 active per token)")

# Training loop
opt_dense = torch.optim.Adam(dense_model.parameters(), lr=1e-3)
opt_moe = torch.optim.Adam(moe_model.parameters(), lr=1e-3)

dense_losses = []
moe_losses = []

for step in range(200):
    x, target = generate_xor_data(32, 16, d)

    # Dense
    dense_model.train()
    pred_dense = dense_model(x)
    loss_dense = F.mse_loss(pred_dense, target)
    opt_dense.zero_grad()
    loss_dense.backward()
    opt_dense.step()
    dense_losses.append(loss_dense.item())

    # MoE
    moe_model.train()
    pred_moe, _ = moe_model(x)
    loss_moe = F.mse_loss(pred_moe, target)
    opt_moe.zero_grad()
    loss_moe.backward()
    opt_moe.step()
    moe_losses.append(loss_moe.item())

# Plot
plt.figure(figsize=(10, 5))
plt.plot(dense_losses, label=f'Dense ({dense_params:,} params)', alpha=0.7)
plt.plot(moe_losses, label=f'MoE 4x ({moe_params:,} params, 1/4 active)', alpha=0.7)
plt.xlabel('Training Step', fontsize=12)
plt.ylabel('MSE Loss', fontsize=12)
plt.title('Dense vs MoE Training Comparison', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal losses -- Dense: {dense_losses[-1]:.4f}, MoE: {moe_losses[-1]:.4f}")

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_24_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. Reflection and Next Steps

### What We Built
- A **Router** that assigns tokens to experts using softmax + top-k selection
- **Expert MLPs** that process only their assigned tokens
- A complete **MoE layer** with sparse computation
- **Expert Choice routing** as an alternative that guarantees balance

### Key Takeaways
1. MoE gives you the capacity of $N \times$ params but the compute of $k \times$ params per token
2. The router is a simple linear layer -- routing complexity is trivial
3. The gather-scatter pattern (collecting tokens for each expert) is the implementation challenge
4. Expert Choice routing avoids load imbalance by letting experts pick tokens

### What Comes Next
In Notebook 2, we will simulate **expert parallelism** -- distributing experts across multiple simulated GPUs with All-to-All communication.